# TV OFF Visual Engine — Cloud Prototype
This notebook is an isolated research prototype. It does not publish to YouTube or access TV OFF secrets.

**Target:** Wan 2.2 TI2V-5B. Use a GPU runtime with at least 24 GB VRAM per the upstream instructions. Free GPU availability is not guaranteed.

In [ ]:
import os, subprocess, sys, json, shutil
from pathlib import Path
print('Python:', sys.version)
print('Disk free GB:', round(shutil.disk_usage('/kaggle/working' if Path('/kaggle/working').exists() else '/content').free/1024**3, 1))
try:
 import torch
 print('Torch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
 if torch.cuda.is_available():
  for i in range(torch.cuda.device_count()):
   p=torch.cuda.get_device_properties(i)
   print(i, p.name, round(p.total_memory/1024**3,1), 'GB VRAM')
except Exception as e: print('Torch diagnostic:', repr(e))


## Runtime gate
Do not download the model unless CUDA is available and the selected GPU has at least 24 GB VRAM. A 16 GB T4/P100 runtime is below the upstream documented minimum for the TI2V-5B 720p command.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU attached. In notebook settings, enable a GPU accelerator and rerun.'
vram_gb=torch.cuda.get_device_properties(0).total_memory/1024**3
assert vram_gb >= 24, f'Only {vram_gb:.1f} GB VRAM detected; Wan2.2 TI2V-5B official 720p command lists >=24 GB. Stop before model download.'
print('GPU gate passed:', torch.cuda.get_device_name(0), f'{vram_gb:.1f} GB')


## Install official Wan2.2 code
This installs upstream dependencies and can take considerable time/storage. The model weights are not included in the Git repository.

In [ ]:
%cd /kaggle/working
!git clone https://github.com/Wan-Video/Wan2.2.git
%cd /kaggle/working/Wan2.2
!pip install -q -r requirements.txt


## Download TI2V-5B weights
The model download is large. Ensure persistent storage and enough free disk before continuing.

In [ ]:
!pip install -q 'huggingface_hub[cli]'
!huggingface-cli download Wan-AI/Wan2.2-TI2V-5B --local-dir ./Wan2.2-TI2V-5B


## Generate the first TV OFF scene
The command below follows the upstream TI2V-5B inference flags. It uses a landscape 1280×704 preview first. Change to 704×1280 for portrait after a successful baseline. Output is research-only; do not publish until model-weight and hosting terms are verified.

In [ ]:
PROMPT = ('Photorealistic cinematic luxury bedroom at midnight, floor-to-ceiling windows, '
          'soft rain outside, warm amber bedside lamps, deep navy shadows, natural wood, '
          'realistic fabrics and architectural proportions. Slow smooth camera dolly forward, '
          'subtle curtain movement and rain, physically plausible reflections, no people, '
          'no text, no logos, no flicker, no geometry changes.')
cmd = [sys.executable, 'generate.py', '--task', 'ti2v-5B', '--size', '1280*704',
       '--ckpt_dir', './Wan2.2-TI2V-5B', '--offload_model', 'True',
       '--convert_model_dtype', '--t5_cpu', '--prompt', PROMPT]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


## Next step
Find the generated MP4 under the upstream output directory, inspect it manually, then run TV OFF's independent FFmpeg validation and create a JSON manifest with prompt, seed, model revision, GPU, VRAM, generation time, output dimensions and SHA-256. Never connect this notebook to the production publisher.